# Run a training script as a command job

**Prince Hands-on Lab – Optimize model training in Azure Machine Learning**

You can use the Python SDK for Azure Machine Learning to submit scripts as **command jobs**. Jobs keep track of the inputs, parameters, outputs and metrics of every training run, so you can compare them later.

## Before you start

You need the **azure-ai-ml** package. The **Python 3.10 - SDK v2** kernel already includes it; the cell below makes sure it is installed and up to date.

> If the output ends with *"you may need to restart the kernel"*, click **Restart kernel** in the toolbar (the ⟳ icon), then continue from the **next** cell.

In [ ]:
%pip install --quiet --upgrade azure-ai-ml azure-identity

## Connect to your workspace

On a compute instance, the workspace details (subscription, resource group, workspace name) are already configured, so `from_config()` finds them automatically.

> If a banner asks you to **Authenticate**, click it and sign in, then run the cell again.

In [ ]:
from lab_helpers import get_ml_client, get_sklearn_environment

ml_client = get_ml_client()

## Choose the training environment

An *environment* is the set of packages installed on the cluster before your script runs. This helper picks a current curated scikit-learn environment in your workspace (or builds a small custom one if none exists), so the job doesn't fail because an old environment was retired.

In [ ]:
sklearn_env = get_sklearn_environment(ml_client)

## Configure and submit a command job

The job runs **src/train-model-parameters.py** on the **aml-cluster** compute cluster, using **diabetes.csv** as the training data.

- `code`: the folder that contains the script. Everything in it is uploaded with the job.
- `command`: exactly what to run. It's the same command you tested in the terminal; `${{inputs.training_data}}` is replaced by the path to the uploaded data.
- `inputs`: the data file (uploaded from this repo) and the regularization rate.
- `environment`: the packages to install on the compute before running the command.
- `compute`: the compute cluster to run on.
- `display_name`: the name of this job. `experiment_name`: the group the job belongs to.

In [ ]:
from azure.ai.ml import command, Input
from azure.ai.ml.constants import AssetTypes

# configure job
job = command(
    code="../src",
    command="python train-model-parameters.py --training_data ${{inputs.training_data}} --reg_rate ${{inputs.reg_rate}}",
    inputs={
        "training_data": Input(type=AssetTypes.URI_FILE, path="../data/diabetes-data/diabetes.csv"),
        "reg_rate": 0.01,
    },
    environment=sklearn_env,
    compute="aml-cluster",
    display_name="diabetes-train-script",
    experiment_name="diabetes-training",
    tags={"model_type": "LogisticRegression"},
)

# submit job
returned_job = ml_client.create_or_update(job)
print("Job name:", returned_job.name)
print("Monitor your job at", returned_job.studio_url)

Click the link above to open the job in the studio. The first job takes **5–10 minutes**: the cluster has to start a node (it scales down to 0 when idle, to save money) and prepare the environment. Later jobs are faster.

## (Optional) Wait for the job here

Run the next cell to stream the job's log into the notebook until it finishes. You can also skip it and watch in the studio.

In [ ]:
ml_client.jobs.stream(returned_job.name)

## Tracking with MLflow

When the script runs as a job it uses MLflow to log:

- the **regularization rate** as a *parameter* → job **Overview** tab, under **Params**
- **Accuracy** and **AUC** as *metrics* → **Metrics** tab
- the **ROC curve** as an *artifact* → **Images** tab, and **Outputs + logs**

The printed output of the script is in **Outputs + logs → user_logs/std_log.txt**.

Next: open **hyperparameter-tuning.ipynb** to train several models in parallel and find the best regularization rate.